# Instrument_Findings — graph-fed consumer

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

print('INSTRUMENT_FINDINGS — graph-fed consumer')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')

INSTRUMENT_FINDINGS — graph-fed consumer
Run: 2026-04-26 17:01


In [2]:
BASE_DIR = Path.cwd().parent          # sdtm-findings-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

DSS_VIEW_FILE = REPO_ROOT / 'consumer-bases' / 'interim' / 'DSS_View.xlsx'
TEST_IDENTITY_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
INSTR_TEST_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Instrument_Test_Identity.xlsx'
INSTR_IDENT_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Instrument_Identity.xlsx'
DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
GRAPH_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'
GRAPH_CT_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph_CT.xlsx'

OUTPUT_DIR = BASE_DIR / 'machine_actionable'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / 'Instrument_Findings.xlsx'

# Instrument-scope domains. No exclusions per behavioural analysis —
# QS, FT, RS form the complete instrument-based Findings set.
SCOPE_DOMAINS = ['QS', 'FT', 'RS']

# Allowed_Units expansion threshold. Codelists with at most this many terms
# are expanded inline; broader codelists (e.g. UNIT, ~950 terms) are passed
# through as the codelist code only.
ALLOWED_UNITS_THRESHOLD = 50

for f, label in [
    (DSS_VIEW_FILE, 'DSS_View'),
    (TEST_IDENTITY_FILE, 'SDTM_Test_Identity'),
    (INSTR_TEST_FILE, 'SDTM_Instrument_Test_Identity'),
    (INSTR_IDENT_FILE, 'SDTM_Instrument_Identity'),
    (DOMAIN_META_FILE, 'SDTM_Domain_Metadata'),
    (GRAPH_FILE, 'COSMoS_Graph'),
    (GRAPH_CT_FILE, 'COSMoS_Graph_CT'),
]:
    if not f.exists():
        raise FileNotFoundError(f'{label} not found: {f}')
    print(f'  {label}: {f.relative_to(REPO_ROOT)}')

print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print(f'  Scope:  {SCOPE_DOMAINS}')
print(f'  Allowed_Units threshold: {ALLOWED_UNITS_THRESHOLD} terms')

  DSS_View: consumer-bases/interim/DSS_View.xlsx
  SDTM_Test_Identity: sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  SDTM_Instrument_Test_Identity: sdtm-test-codes/machine_actionable/SDTM_Instrument_Test_Identity.xlsx
  SDTM_Instrument_Identity: sdtm-test-codes/machine_actionable/SDTM_Instrument_Identity.xlsx
  SDTM_Domain_Metadata: sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx
  COSMoS_Graph: cosmos-graph/interim/COSMoS_Graph.xlsx
  COSMoS_Graph_CT: cosmos-graph/interim/COSMoS_Graph_CT.xlsx
  Output: sdtm-findings-graph/machine_actionable/Instrument_Findings.xlsx
  Scope:  ['QS', 'FT', 'RS']
  Allowed_Units threshold: 50 terms


## 2. Load inputs

In [3]:
view_ti = pd.read_excel(DSS_VIEW_FILE, sheet_name='Test_Identity', dtype=str).fillna('')
view_ms = pd.read_excel(DSS_VIEW_FILE, sheet_name='Measurement_Specs', dtype=str).fillna('')
ref_tests = pd.read_excel(TEST_IDENTITY_FILE, sheet_name='Test Codes', dtype=str).fillna('')
instr_tests = pd.read_excel(INSTR_TEST_FILE, sheet_name='Test Codes', dtype=str).fillna('')
instr_ident = pd.read_excel(INSTR_IDENT_FILE, sheet_name='Instruments', dtype=str).fillna('')
domain_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')
ct_terms = pd.read_excel(GRAPH_CT_FILE, sheet_name='CodelistTerms', dtype=str).fillna('')
graph_bc = pd.read_excel(GRAPH_FILE, sheet_name='BC', dtype=str).fillna('')
graph_bc_parents = pd.read_excel(GRAPH_FILE, sheet_name='BC_Parents', dtype=str).fillna('')
graph_bc_cats = pd.read_excel(GRAPH_FILE, sheet_name='BC_Categories', dtype=str).fillna('')
graph_dss = pd.read_excel(GRAPH_FILE, sheet_name='DSS', dtype=str).fillna('')

print(f'DSS_View Test_Identity:           {len(view_ti):>6,} rows')
print(f'DSS_View Measurement_Specs:       {len(view_ms):>6,} rows')
print(f'SDTM_Test_Identity:               {len(ref_tests):>6,} rows')
print(f'SDTM_Instrument_Test_Identity:    {len(instr_tests):>6,} rows')
print(f'SDTM_Instrument_Identity:         {len(instr_ident):>6,} rows')
print(f'SDTM_Domain_Metadata:             {len(domain_meta):>6,} rows')
print(f'COSMoS_Graph BC:                  {len(graph_bc):>6,} rows')
print(f'COSMoS_Graph BC_Parents:          {len(graph_bc_parents):>6,} rows')
print(f'COSMoS_Graph BC_Categories:       {len(graph_bc_cats):>6,} rows')
print(f'COSMoS_Graph DSS:                 {len(graph_dss):>6,} rows')
print(f'COSMoS_Graph_CT CodelistTerms:    {len(ct_terms):>6,} rows')

DSS_View Test_Identity:              729 rows
DSS_View Measurement_Specs:        1,326 rows
SDTM_Test_Identity:                5,885 rows
SDTM_Instrument_Test_Identity:    10,311 rows
SDTM_Instrument_Identity:            359 rows
SDTM_Domain_Metadata:                 57 rows
COSMoS_Graph BC:                   1,345 rows
COSMoS_Graph BC_Parents:           1,073 rows
COSMoS_Graph BC_Categories:        4,096 rows
COSMoS_Graph DSS:                  1,326 rows
COSMoS_Graph_CT CodelistTerms:    17,523 rows


## 3. Build Test_Identity

In [4]:
# Three universes contribute to instrument Test_Identity:
#   1. SDTM_Instrument_Test_Identity — per-instrument-codelist test codes (the canonical case).
#   2. SDTM_Test_Identity scoped to QS/FT/RS — domain-level codelists (RS multi-framework umbrella).
#   3. DSS_View pinned TESTCDs not in either source — LZZT example items.
# These are disjoint by construction (their codelist anchors differ); union gives the full universe.

# Universe 1 — instrument codelists.
u1 = instr_tests[instr_tests['Domain'].isin(SCOPE_DOMAINS)].copy()
u1['ct_source'] = 'instrument_codelist'
u1 = u1.rename(columns={'Domain': 'In_Scope_Domains'})
u1['SDTM_Domains'] = u1['In_Scope_Domains']
print(f'Universe 1 — instrument_codelist: {len(u1):,} rows')

# Universe 2 — domain-level RSTESTCD-style codelists from the wider SDTM_Test_Identity.
def in_scope(domains_str):
    if not domains_str:
        return False
    return bool({d.strip() for d in domains_str.split(';')} & set(SCOPE_DOMAINS))

u2 = ref_tests[ref_tests['SDTM_Domains'].apply(in_scope)].copy()
u2['ct_source'] = 'domain_codelist'
# In_Scope_Domains: intersection of SDTM_Domains and SCOPE_DOMAINS
u2['In_Scope_Domains'] = u2['SDTM_Domains'].apply(
    lambda s: ';'.join(sorted({d.strip() for d in s.split(';')} & set(SCOPE_DOMAINS)))
)
u2['Instrument'] = ''
u2['Codelist_TESTCD'] = ''
print(f'Universe 2 — domain_codelist:     {len(u2):,} rows')

# Universe 3 — LZZT example TESTCDs in DSS_View not in either u1 or u2.
scope_dss_tcs = set(view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)]['TESTCD_value'])
u3_tcs = scope_dss_tcs - set(u1['TESTCD']) - set(u2['TESTCD'])
if u3_tcs:
    u3_src = view_ms[view_ms['TESTCD_value'].isin(u3_tcs)]\
        [['TESTCD_value', 'TEST_value', 'TESTCD_ncit', 'domain']].drop_duplicates('TESTCD_value')
    u3 = pd.DataFrame({
        'NCIt_Code': u3_src['TESTCD_ncit'].values,
        'TESTCD':    u3_src['TESTCD_value'].values,
        'TEST':      u3_src['TEST_value'].values,
        'In_Scope_Domains':    u3_src['domain'].values,
        'SDTM_Domains':        u3_src['domain'].values,
        'Instrument': '',
        'Codelist_TESTCD': '',
        'NCIt_Preferred_Term': '',
        'NCIt_Synonyms': '',
        'NCIt_Definition': '',
        'UMLS_CUI': '',
        'NCIm_CUI': '',
        'ct_source': 'lzzt_example',
    })
else:
    u3 = pd.DataFrame()
print(f'Universe 3 — lzzt_example:        {len(u3):,} rows')

# Union — preserve column union; align dtypes as str.
ti_universe = pd.concat([u1, u2, u3], ignore_index=True, sort=False).fillna('')
print(f'Test_Identity universe (union):   {len(ti_universe):,} rows')

Universe 1 — instrument_codelist: 10,257 rows
Universe 2 — domain_codelist:     48 rows
Universe 3 — lzzt_example:        4 rows
Test_Identity universe (union):   10,309 rows


In [5]:
# Join SDTM_Instrument_Identity to get instrument/container anchors per Codelist_TESTCD.
ident_cols = ['Codelist_TESTCD', 'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term',
              'Instrument_Match_Method', 'Container_NCIt_Code', 'Container_NCIt_Preferred_Term']
ident_join = instr_ident[ident_cols].copy()

ti = ti_universe.merge(ident_join, on='Codelist_TESTCD', how='left').fillna('')

# For domain_codelist and lzzt_example rows, codelist anchor doesn't apply — set match method.
ti.loc[ti['ct_source'] == 'domain_codelist', 'Instrument_Match_Method'] = 'not_applicable'
ti.loc[ti['ct_source'] == 'lzzt_example',    'Instrument_Match_Method'] = 'not_applicable'

print(f'After SDTM_Instrument_Identity join: {len(ti):,} rows')
print('\nInstrument_Match_Method distribution:')
print(ti['Instrument_Match_Method'].value_counts(dropna=False))

After SDTM_Instrument_Identity join: 10,309 rows

Instrument_Match_Method distribution:
Instrument_Match_Method
exact             4943
UNMATCHED         2890
normalized        1608
suffix-strip       816
not_applicable      52
Name: count, dtype: int64


In [6]:
# COSMoS coverage from DSS_View Measurement_Specs, filtered to scope domains.
view_ms_scope = view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)].copy()
print(f'In-scope DSS rows in DSS_View: {len(view_ms_scope):,}')

def _join_unique(s):
    return '; '.join(sorted({v for v in s if v}))

cosmos_cov = view_ms_scope.groupby('TESTCD_value', sort=True).agg(
    DSS_Count=('ds_id', 'nunique'),
    BC_Count=('bc_id', 'nunique'),
    DS_Codes=('ds_short_name', _join_unique),
    BC_IDs=('bc_id', _join_unique),
).reset_index().rename(columns={'TESTCD_value': 'TESTCD'})

cosmos_cov['DSS_Count'] = cosmos_cov['DSS_Count'].astype(str)
cosmos_cov['BC_Count'] = cosmos_cov['BC_Count'].astype(str)
print(f'TESTCDs with COSMoS DSSs in scope: {len(cosmos_cov):,}')

In-scope DSS rows in DSS_View: 175
TESTCDs with COSMoS DSSs in scope: 169


In [7]:
# NCIt_Code_Conflict and NCIt_Reference_Disagree flags.
# These flags are computed in DSS_View Test_Identity over the COSMoS-pinned subset.
# Carry through where present; default empty for non-pinned TESTCDs.
flags = view_ti[['TESTCD', 'NCIt_Code_Conflict']].drop_duplicates()
# NCIt_Reference_Disagree exists in DSS_View Test_Identity
if 'NCIt_Reference_Disagree' in view_ti.columns:
    flags = view_ti[['TESTCD', 'NCIt_Code_Conflict', 'NCIt_Reference_Disagree']].drop_duplicates()
else:
    flags['NCIt_Reference_Disagree'] = ''
print(f'TESTCDs with conflict flags from DSS_View: {len(flags):,}')

TESTCDs with conflict flags from DSS_View: 729


In [8]:
# Has_DSS membership and merge of cosmos_cov + flags.
ti['Has_DSS'] = ti['TESTCD'].isin(cosmos_cov['TESTCD']).map({True: 'Yes', False: 'No'})
ti = ti.merge(cosmos_cov, on='TESTCD', how='left').fillna('')
ti.loc[ti['DSS_Count'] == '', 'DSS_Count'] = '0'
ti.loc[ti['BC_Count'] == '', 'BC_Count'] = '0'
ti = ti.merge(flags, on='TESTCD', how='left').fillna('')

# Drop the auxiliary 'Instrument' column from Universe 1 — its content is captured in
# the chocolate Instrument_NCIt_Preferred_Term column from SDTM_Instrument_Identity.
ti = ti.drop(columns=['Instrument'], errors='ignore')

TI_COLS = [
    'TESTCD', 'NCIt_Code', 'In_Scope_Domains', 'SDTM_Domains', 'ct_source',
    'TEST', 'NCIt_Preferred_Term', 'NCIt_Synonyms', 'NCIt_Definition', 'UMLS_CUI', 'NCIm_CUI',
    'Codelist_TESTCD',
    'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term', 'Instrument_Match_Method',
    'Container_NCIt_Code', 'Container_NCIt_Preferred_Term',
    'Has_DSS', 'DSS_Count', 'BC_Count', 'DS_Codes', 'BC_IDs',
    'NCIt_Code_Conflict', 'NCIt_Reference_Disagree',
]

missing = [c for c in TI_COLS if c not in ti.columns]
if missing:
    raise RuntimeError(f'Missing expected Test_Identity columns: {missing}')
ti_final = ti[TI_COLS].copy()
ti_final = ti_final.sort_values(['ct_source', 'In_Scope_Domains', 'TESTCD']).reset_index(drop=True)
print(f'Test_Identity: {len(ti_final):,} rows x {len(ti_final.columns)} cols')
print(f'  Has_DSS=Yes: {(ti_final["Has_DSS"] == "Yes").sum():,}')
print(f'  Coverage gap (Has_DSS=No): {(ti_final["Has_DSS"] == "No").sum():,}')
print(f'  ct_source distribution:')
for k, v in ti_final['ct_source'].value_counts().items():
    print(f'    {k}: {v:,}')

Test_Identity: 10,309 rows x 24 cols


  Has_DSS=Yes: 169
  Coverage gap (Has_DSS=No): 10,140
  ct_source distribution:
    instrument_codelist: 10,257
    domain_codelist: 48
    lzzt_example: 4


## 4. Build Measurement_Specs

In [9]:
ms = view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)].copy()
print(f'Measurement_Specs rows in scope: {len(ms):,}')
print(f'  Domains present:           {sorted(ms["domain"].unique())}')
print(f'  Domains with no DSSs yet:  {sorted(set(SCOPE_DOMAINS) - set(ms["domain"].unique()))}')

Measurement_Specs rows in scope: 175
  Domains present:           ['FT', 'QS', 'RS']
  Domains with no DSSs yet:  []


In [10]:
# Observation_Class join.
ms = ms.merge(
    domain_meta[['Domain', 'Observation_Class']].rename(columns={'Domain': 'domain'}),
    on='domain', how='left',
  ).fillna('')

# LOINC_BC: coalesce the two source URI variants. The graph projects the URI
# inconsistency in the BC Coding list verbatim — see cosmos-graph/docs/COSMoS_Graph.md.
LOINC_BC_SOURCES = ['Coding_http://loinc.org/', 'Coding_https://loinc.org']
def _coalesce(row):
    for col in LOINC_BC_SOURCES:
        if col in row and row[col]:
            return row[col]
    return ''
ms['LOINC_BC'] = ms.apply(_coalesce, axis=1)

In [11]:
# Allowed_Units: conditional expansion of ORRESU_codelist when narrow enough
# that per-row expansion is signal, not noise.
codelist_size = ct_terms.groupby('codelist_submission_value')['term_submission_value'].nunique()
narrow_codelists = set(codelist_size[codelist_size <= ALLOWED_UNITS_THRESHOLD].index)
print(f'Codelists with <= {ALLOWED_UNITS_THRESHOLD} terms: {len(narrow_codelists):,}')

terms_by_codelist = {}
for cl, group in ct_terms.groupby('codelist_submission_value'):
    if cl in narrow_codelists:
        terms = sorted({t for t in group['term_submission_value'] if t})
        terms_by_codelist[cl] = '; '.join(terms)

ms['Allowed_Units'] = ms['ORRESU_codelist'].map(terms_by_codelist).fillna('')
expanded = (ms['Allowed_Units'] != '').sum()
print(f'Rows with Allowed_Units expansion: {expanded:,}')
if expanded:
    fired_codelists = sorted({cl for cl in ms['ORRESU_codelist'] if cl in narrow_codelists})
    print(f'  Fired codelists: {fired_codelists}')

Codelists with <= 50 terms: 244
Rows with Allowed_Units expansion: 1
  Fired codelists: ['AGEU']


In [12]:
# Rename DSS_View link-key columns for join symmetry with Test_Identity.
ms = ms.rename(columns={
    'TESTCD_value': 'TESTCD',
    'TEST_value':   'TEST',
    'TESTCD_ncit':  'TESTCD_NCIt',
    'LOINC_value':  'LOINC_DSS',
    'TESTCD_codelist': 'Codelist_TESTCD',
})

In [13]:
# ct_source assignment — classify by TESTCD presence in source files.
# TESTCD is the universal key; Codelist_TESTCD is an enrichment we backfill below.
#   instrument_codelist — TESTCD in SDTM_Instrument_Test_Identity (per-instrument codelist).
#   domain_codelist     — TESTCD in SDTM_Test_Identity (domain-level RS umbrella).
#   lzzt_example        — TESTCD in neither standards file (CDISC Pilot only).
instr_testcd_set = set(instr_tests['TESTCD'])
domain_codelist_tcs = set(ref_tests[ref_tests['SDTM_Domains'].apply(in_scope)]['TESTCD'])

def classify_ct_source(testcd):
    if testcd in instr_testcd_set:
        return 'instrument_codelist'
    if testcd in domain_codelist_tcs:
        return 'domain_codelist'
    return 'lzzt_example'

ms['ct_source'] = ms['TESTCD'].apply(classify_ct_source)

# Backfill missing Codelist_TESTCD from SDTM_Instrument_Test_Identity using TESTCD.
# DSS_View doesn't always carry Codelist_TESTCD in the pin pivot (e.g., ADCCMD).
testcd_to_codelist = dict(zip(instr_tests['TESTCD'], instr_tests['Codelist_TESTCD']))
missing_mask = (ms['Codelist_TESTCD'] == '') & (ms['ct_source'] == 'instrument_codelist')
n_backfilled = missing_mask.sum()
if n_backfilled:
    ms.loc[missing_mask, 'Codelist_TESTCD'] = ms.loc[missing_mask, 'TESTCD'].map(testcd_to_codelist).fillna('')
    print(f'Backfilled Codelist_TESTCD for {n_backfilled} rows missing it in DSS_View')

print('ct_source distribution on Measurement_Specs:')
print(ms['ct_source'].value_counts())

Backfilled Codelist_TESTCD for 1 rows missing it in DSS_View
ct_source distribution on Measurement_Specs:
ct_source
instrument_codelist    150
domain_codelist         21
lzzt_example             4
Name: count, dtype: int64


In [14]:
# Join SDTM_Instrument_Identity for chocolate (C20993) and copper (C211913) anchors.
ident_join = instr_ident[['Codelist_TESTCD', 'Instrument_NCIt_Code',
                          'Instrument_NCIt_Preferred_Term', 'Instrument_Match_Method',
                          'Container_NCIt_Code', 'Container_NCIt_Preferred_Term']].copy()
ms = ms.merge(ident_join, on='Codelist_TESTCD', how='left').fillna('')

# For domain_codelist and lzzt_example rows, anchor join doesn't apply.
ms.loc[ms['ct_source'] == 'domain_codelist', 'Instrument_Match_Method'] = 'not_applicable'
ms.loc[ms['ct_source'] == 'lzzt_example',    'Instrument_Match_Method'] = 'not_applicable'
print('Instrument_Match_Method on Measurement_Specs:')
print(ms['Instrument_Match_Method'].value_counts())

Instrument_Match_Method on Measurement_Specs:
Instrument_Match_Method
exact             97
not_applicable    25
suffix-strip      18
normalized        18
UNMATCHED         17
Name: count, dtype: int64


In [15]:
# Join cosmos-graph BC_Parents for parent_bc_id.
# DSS_View carries bc_parent_label and bc_type but not parent_bc_id; we add it for join symmetry.
parent_extra = graph_bc_parents[['bc_id', 'parent_bc_id']].drop_duplicates()
ms = ms.merge(parent_extra, on='bc_id', how='left').fillna('')
print(f'bc_type distribution on Measurement_Specs (from DSS_View):')
print(ms['bc_type'].value_counts(dropna=False))

bc_type distribution on Measurement_Specs (from DSS_View):
bc_type
full    175
Name: count, dtype: int64


In [16]:
# PIN_SCHEMA — instrument-specific. core = first-order structural pins; optional = qualifiers.
# core is intentionally minimal for instrument because the structural payload lives in the
# BC hierarchy and the codelist anchors, not in pins. Empty core would also be a valid
# signal — TESTCD/TEST/CAT are kept as core because they are universally pinned and
# fundamental to the row's identity at SDTM level.
PIN_SCHEMA = {
    'core':     ['CAT'],
    'optional': ['SCAT', 'ORRESU', 'STRESU', 'EVAL', 'EVINTX', 'EVLINT',
                 'METHOD', 'LOC', 'ANTXLO', 'ANTXHI', 'ANVLLO', 'ANVLHI'],
    'exclude':  ['TESTCD', 'TEST', 'LOINC'],
}

def fires(remainder):
    for suffix in ('_value', '_ncit', '_codelist'):
        col = f'{remainder}{suffix}'
        if col in ms.columns and (ms[col] != '').any():
            return True
    return False

declared = set(PIN_SCHEMA['core']) | set(PIN_SCHEMA['optional']) | set(PIN_SCHEMA['exclude'])
all_candidate_remainders = sorted({c.rsplit('_', 1)[0] for c in ms.columns if c.endswith('_value')})

pins_to_emit = []
notes = []

for r in PIN_SCHEMA['core']:
    if f'{r}_value' not in ms.columns:
        notes.append(f"WARN: core pin '{r}' has no columns in DSS_View — schema likely needs updating.")
        continue
    pins_to_emit.append(r)
    if not fires(r):
        notes.append(f"INFO: core pin '{r}' not firing on this scope — schema preserved, column will be empty.")

for r in PIN_SCHEMA['optional']:
    if fires(r):
        pins_to_emit.append(r)

for r in all_candidate_remainders:
    if r in declared:
        continue
    if fires(r):
        n_rows = (ms[f'{r}_value'] != '').sum()
        notes.append(f"WARN: undeclared pin '{r}' firing on {n_rows} rows — add to PIN_SCHEMA core/optional/exclude.")
        pins_to_emit.append(r)

for note in notes:
    print(note)
if notes:
    print()

print(f'Pins to emit ({len(pins_to_emit)}, in order): {pins_to_emit}')
print(f"  Core (always present):  {[r for r in pins_to_emit if r in PIN_SCHEMA['core']]}")
print(f"  Optional (firing-only): {[r for r in pins_to_emit if r in PIN_SCHEMA['optional']]}")
print(f"  Undeclared (firing):    {[r for r in pins_to_emit if r not in declared]}")

Pins to emit (13, in order): ['CAT', 'SCAT', 'ORRESU', 'STRESU', 'EVAL', 'EVINTX', 'EVLINT', 'METHOD', 'LOC', 'ANTXLO', 'ANTXHI', 'ANVLLO', 'ANVLHI']
  Core (always present):  ['CAT']
  Optional (firing-only): ['SCAT', 'ORRESU', 'STRESU', 'EVAL', 'EVINTX', 'EVLINT', 'METHOD', 'LOC', 'ANTXLO', 'ANTXHI', 'ANVLLO', 'ANVLHI']
  Undeclared (firing):    []


In [17]:
# Assemble final column order — DSS identity, ct_source, link keys, BC identity,
# instrument anchor, container anchor, LOINC pair, pins, Allowed_Units.
identity_cols = [
    'domain', 'Observation_Class', 'ds_id', 'bc_id', 'parent_bc_id',
    'ct_source',
    'TESTCD', 'TEST', 'TESTCD_NCIt', 'Codelist_TESTCD',
    'bc_short_name', 'bc_definition', 'bc_categories', 'bc_parent_label',
    'bc_hierarchy_path', 'bc_type', 'result_scales', 'bc_ncit_code',
    'ds_short_name', 'sdtmig_start_version', 'sdtmig_end_version',
    'package_date', 'source',
]

anchor_cols = [
    'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term', 'Instrument_Match_Method',
    'Container_NCIt_Code', 'Container_NCIt_Preferred_Term',
]

loinc_cols = ['LOINC_DSS', 'LOINC_BC']

pin_cols = []
for r in pins_to_emit:
    for suffix in ('_value', '_ncit', '_codelist'):
        col = f'{r}{suffix}'
        if col in ms.columns:
            pin_cols.append(col)

tail_cols = ['Allowed_Units']

MS_COLS = identity_cols + anchor_cols + loinc_cols + pin_cols + tail_cols

missing = [c for c in MS_COLS if c not in ms.columns]
if missing:
    raise RuntimeError(f'Missing expected Measurement_Specs columns: {missing}')

ms_final = ms[MS_COLS].copy()
ms_final = ms_final.sort_values(['domain', 'bc_id', 'TESTCD', 'ds_id']).reset_index(drop=True)
print(f'Measurement_Specs: {len(ms_final):,} rows x {len(ms_final.columns)} cols')
print(f'  Identity: {len(identity_cols)}  Anchors: {len(anchor_cols)}  '
      f'LOINC: {len(loinc_cols)}  Pins: {len(pin_cols)}  Tail: {len(tail_cols)}')

Measurement_Specs: 175 rows x 70 cols
  Identity: 23  Anchors: 5  LOINC: 2  Pins: 39  Tail: 1


## 5. Build BC_Categories

In [18]:
# BC_Categories — (bc_id, category) edge list.
# Scope: in-scope DSS-bearing BCs (the 144 'full' BCs), plus instrument-level companions
# reached by category-string overlap (the ~120 'full_no_ds' BCs that share categories).
# This supports Linda's '7 BCs of 6MWT' search use case — instrument-level BC + items.

# Step 1: in-scope DSS-bearing BCs.
scope_dss_bcs = set(graph_dss[graph_dss['domain'].isin(SCOPE_DOMAINS)]['bc_id'])
print(f'In-scope DSS-bearing BCs (full): {len(scope_dss_bcs):,}')

# Step 2: their category strings.
scope_cats = set(graph_bc_cats[graph_bc_cats['bc_id'].isin(scope_dss_bcs)]['category'])
print(f'Categories used by in-scope DSS-bearing BCs: {len(scope_cats):,}')

# Step 3: any BC that shares at least one of those categories.
scope_companion_bcs = set(graph_bc_cats[graph_bc_cats['category'].isin(scope_cats)]['bc_id'])
scope_bc_ids = scope_dss_bcs | scope_companion_bcs
print(f'Total scope BCs (in-scope DSS + companions): {len(scope_bc_ids):,}')

# Filter BC_Categories to scope.
bc_cats = graph_bc_cats[graph_bc_cats['bc_id'].isin(scope_bc_ids)].copy()

# Join bc_short_name and bc_type.
bc_cats = bc_cats.merge(
    graph_bc[['bc_id', 'bc_short_name', 'bc_type']].drop_duplicates('bc_id'),
    on='bc_id', how='left'
).fillna('')

# domains: for full BCs, the SDTM domain from DSS; for full_no_ds, derive from
# DSS-bearing companions sharing categories (most-frequent companion domain).
bc_to_domain_full = graph_dss[graph_dss['domain'].isin(SCOPE_DOMAINS)]\
    .groupby('bc_id')['domain'].apply(lambda s: ';'.join(sorted(set(s)))).to_dict()

def derive_domain(row):
    bc_id = row['bc_id']
    if bc_id in bc_to_domain_full:
        return bc_to_domain_full[bc_id]
    # full_no_ds — find DSS-bearing companions with this category and aggregate their domains.
    cat = row['category']
    companion_bcs = set(graph_bc_cats[graph_bc_cats['category'] == cat]['bc_id']) & scope_dss_bcs
    doms = sorted({bc_to_domain_full[b] for b in companion_bcs if b in bc_to_domain_full})
    return ';'.join(doms)

bc_cats['domains'] = bc_cats.apply(derive_domain, axis=1)

# in_dss_scope flag: True if bc_id has a DSS in QS/FT/RS.
bc_cats['in_dss_scope'] = bc_cats['bc_id'].isin(scope_dss_bcs).map({True: 'Yes', False: 'No'})

# Final column order — preserve cosmos-graph BC_Categories row order within each bc_id.
BC_CATS_COLS = ['bc_id', 'bc_short_name', 'bc_type', 'category', 'domains', 'in_dss_scope']
bc_cats_final = bc_cats[BC_CATS_COLS].copy().reset_index(drop=True)
print(f'BC_Categories: {len(bc_cats_final):,} rows x {len(bc_cats_final.columns)} cols')
print(f'  Distinct BCs:                 {bc_cats_final["bc_id"].nunique():,}')
print(f'  Distinct categories:          {bc_cats_final["category"].nunique():,}')
print(f'  bc_type distribution:')
for k, v in bc_cats_final.drop_duplicates("bc_id")["bc_type"].value_counts().items():
    print(f'    {k}: {v:,} BCs')

In-scope DSS-bearing BCs (full): 169
Categories used by in-scope DSS-bearing BCs: 99
Total scope BCs (in-scope DSS + companions): 325


BC_Categories: 1,906 rows x 6 cols
  Distinct BCs:                 325
  Distinct categories:          139
  bc_type distribution:
    full: 179 BCs
    full_no_ds: 146 BCs


## 6. Build BC_Parents

In [19]:
# BC_Parents — (bc_id, parent_bc_id) edge list.
# Scope: same set as BC_Categories (in-scope DSS-bearing + companions), plus parent-closure
# so each scope BC's parent chain is walkable to top class.

# Walk parents recursively from scope_bc_ids to gather all ancestors.
parent_of = dict(zip(graph_bc_parents['bc_id'], graph_bc_parents['parent_bc_id']))

all_in_scope = set(scope_bc_ids)
for b in list(scope_bc_ids):
    cur = b
    visited = {cur}
    while True:
        nxt = parent_of.get(cur, '')
        if not nxt or nxt in visited:
            break
        all_in_scope.add(nxt)
        visited.add(nxt)
        cur = nxt

print(f'BCs in scope (with parent closure): {len(all_in_scope):,}')

# Filter BC_Parents to edges where bc_id is in scope (parent will be in scope too via closure).
bc_par = graph_bc_parents[graph_bc_parents['bc_id'].isin(all_in_scope)].copy()

# Join child bc_short_name and bc_type.
bc_par = bc_par.merge(
    graph_bc[['bc_id', 'bc_short_name', 'bc_type']].drop_duplicates('bc_id'),
    on='bc_id', how='left'
).fillna('')

# Join parent bc_type.
parent_meta = graph_bc[['bc_id', 'bc_type']].drop_duplicates('bc_id').rename(
    columns={'bc_id': 'parent_bc_id', 'bc_type': 'parent_bc_type'}
)
bc_par = bc_par.merge(parent_meta, on='parent_bc_id', how='left').fillna('')

# in_dss_scope flag.
bc_par['in_dss_scope'] = bc_par['bc_id'].isin(scope_dss_bcs).map({True: 'Yes', False: 'No'})

BC_PAR_COLS = ['bc_id', 'bc_short_name', 'bc_type', 'parent_bc_id',
               'parent_bc_short_name', 'parent_bc_type', 'in_dss_scope']
bc_par_final = bc_par[BC_PAR_COLS].copy()
bc_par_final = bc_par_final.sort_values(['bc_id', 'parent_bc_id']).reset_index(drop=True)
print(f'BC_Parents: {len(bc_par_final):,} rows x {len(bc_par_final.columns)} cols')
print(f'  Edges where child has DSS in scope: {(bc_par_final["in_dss_scope"] == "Yes").sum():,}')
print(f'  Edges where child is full_no_ds:     {(bc_par_final["bc_type"] == "full_no_ds").sum():,}')

BCs in scope (with parent closure): 358


BC_Parents: 329 rows x 7 cols
  Edges where child has DSS in scope: 169
  Edges where child is full_no_ds:     150


## 7. Write workbook

In [20]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

# Repo header colour convention.
GREEN_HEADER     = PatternFill('solid', fgColor='548235')   # TESTCD / SDTM-CT-side
YELLOW_HEADER    = PatternFill('solid', fgColor='FFD700')   # COSMoS-side
GREY_HEADER      = PatternFill('solid', fgColor='808080')   # keys, flags, aggregations
CHOCOLATE_HEADER = PatternFill('solid', fgColor='5B3A29')   # instrument anchor (NCIt C20993)
COPPER_HEADER    = PatternFill('solid', fgColor='CC7A3E')   # container anchor (NCIt C211913)


def write_sheet(ws, df, header_fills, col_widths):
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'

print('Style helpers ready (5 fills: green / yellow / grey / chocolate / copper).')

Style helpers ready (5 fills: green / yellow / grey / chocolate / copper).


In [21]:
wb = Workbook()

# ── ReadMe sheet ──
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('Instrument_Findings — graph-fed consumer', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    ('Notebook: sdtm-findings-graph/notebooks/Instrument_Findings.ipynb', readme_font),
    ('Track:    sdtm-findings-graph (graph-fed Findings consumer)', readme_font),
    ('Inputs:', readme_font),
    ('  consumer-bases/interim/DSS_View.xlsx (Test_Identity, Measurement_Specs)', readme_font),
    ('  sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx (Test Codes)', readme_font),
    ('  sdtm-test-codes/machine_actionable/SDTM_Instrument_Test_Identity.xlsx', readme_font),
    ('  sdtm-test-codes/machine_actionable/SDTM_Instrument_Identity.xlsx', readme_font),
    ('  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx (Domains)', readme_font),
    ('  cosmos-graph/interim/COSMoS_Graph.xlsx (BC, BC_Parents, BC_Categories, DSS)', readme_font),
    ('  cosmos-graph/interim/COSMoS_Graph_CT.xlsx (CodelistTerms — for Allowed_Units)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('Instrument-based Findings — questionnaires, functional tests, response scales.', readme_font),
    ('In scope: QS, FT, RS. No exclusions.', readme_font),
    ('Behavioural rationale: cosmos-bc-dss/docs/COSMoS_Behavioural_Analysis.md', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Test_Identity — one row per TESTCD scoped to instrument domains. Universe is', readme_font),
    ('  the union of SDTM_Instrument_Test_Identity (per-instrument codelists),', readme_font),
    ('  SDTM_Test_Identity (domain-level RSTESTCD umbrella), and LZZT example items.', readme_font),
    ('  ct_source flags which universe a row comes from. Has_DSS flags whether COSMoS', readme_font),
    ('  pins the TESTCD with a DSS.', readme_font),
    ('Measurement_Specs — one row per DSS in QS, FT, RS. Carries pin pivot, BC identity,', readme_font),
    ('  instrument anchor (NCIt C20993, chocolate), container anchor (NCIt C211913,', readme_font),
    ('  copper), LOINC dual-grain, and Allowed_Units expansion. PIN_SCHEMA core = [CAT];', readme_font),
    ('  optional pins emitted only when firing on the in-scope subset.', readme_font),
    ('BC_Categories — (BC, category) edge list. Includes both DSS-bearing item BCs (full)', readme_font),
    ('  and instrument-level companion BCs (full_no_ds) reached by category overlap, so', readme_font),
    ('  searches like "all BCs tagged 6MWT" return the instrument BC + its 6 item BCs.', readme_font),
    ('  Categories are heterogeneous tags — domain codes, class labels, codelist values,', readme_font),
    ('  acronyms, instrument long names. Faithful pass-through, no auto-typing.', readme_font),
    ('BC_Parents — (BC, parent) edge list. Faithful BC parent-child traversal. Both', readme_font),
    ('  parallel chains visible: items → wrapper → container codelist → top class, and', readme_font),
    ('  instrument-level BC → top class (the two NCIt-disjoint chains noted in the case', readme_font),
    ('  studies). Parent chain closed to the top so traversal is complete in-sheet.', readme_font),
    ('', None),
    ('PARALLEL BC CHAINS', section_font),
    ('NCIt models instrument-as-tool (C20993) and question containers (C211913) in', readme_font),
    ('disjoint trees. Item BCs roll up via Question wrappers (C115409 for 6MWT) — NOT', readme_font),
    ('via the instrument-level BC (C115789 for 6MWT). The two chains connect only', readme_font),
    ('through shared BC_Categories tags. See:', readme_font),
    ('  docs/6MWT_NCIt_Story.html  — disjoint trees + 28% C20993 gap', readme_font),
    ('  docs/6MWT_COSMoS_Story.html — what COSMoS adds for composition+classification', readme_font),
    ('', None),
    ('CONSUMER-OWED DECISIONS APPLIED', section_font),
    ('LOINC dual-grain: LOINC_DSS (DSS-pinned) and LOINC_BC (BC-grain, coalesced from', readme_font),
    ('  the two URI variants). Both typically empty for instrument.', readme_font),
    ('Allowed_Units expansion: applied. ORRESU_codelist is expanded against', readme_font),
    (f'  CodelistTerms only when the codelist has <= {ALLOWED_UNITS_THRESHOLD} terms.', readme_font),
    ('TESTCD universe widening: yes — Test_Identity uses the wider three-source union', readme_font),
    ('  (SDTM_Instrument_Test_Identity ∪ SDTM_Test_Identity scoped ∪ LZZT), not just', readme_font),
    ('  COSMoS-pinned TESTCDs.', readme_font),
    ('Domain class: Observation_Class joined per Measurement_Specs row.', readme_font),
    ('ct_source discriminator: instrument_codelist / domain_codelist / lzzt_example.', readme_font),
    ('Instrument_Match_Method values: exact / suffix-strip / normalized / UNMATCHED', readme_font),
    ('  / not_applicable. UNMATCHED rows have a codelist but no NCIt instrument anchor', readme_font),
    ('  (the 28% gap — versioned variants, respondent formats, missing concepts).', readme_font),
    ('  not_applicable flags rows where no instrument codelist applies (depth-2 RS, LZZT).', readme_font),
    ('', None),
    ('CONFLICT FLAGS', section_font),
    ('NCIt_Code_Conflict (Test_Identity) — multiple NCIt concepts resolve from the', readme_font),
    ('  same TESTCD across the graph.', readme_font),
    ('NCIt_Reference_Disagree (Test_Identity) — NCIt at TESTCD grain disagrees', readme_font),
    ('  between graph (COSMoS pin) and reference (SDTM_Test_Identity / NCI EVS).', readme_font),
    ('Validation belongs in a separate step, not in this consumer projection.', readme_font),
    ('', None),
    ('HEADER COLOUR CONVENTION', section_font),
    ('Green     = TESTCD / SDTM-CT-side columns (NCI EVS reference identity).', readme_font),
    ('Yellow    = COSMoS-side columns (BC, DSS, pin pivot).', readme_font),
    ('Chocolate = Instrument anchor — NCIt class C20993 (instrument-as-tool).', readme_font),
    ('Copper    = Container anchor — NCIt class C211913 (question container/codelist).', readme_font),
    ('Grey      = keys, conflict flags, aggregation columns.', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Instrument sub-type — third and final build of the graph-fed Findings consumer.', readme_font),
    ('Parallel to the legacy sdtm-findings/Instrument_Findings.xlsx, which reads', readme_font),
    ('cosmos-bc-dss/interim/COSMoS_BC_DSS.xlsx. The legacy retires once specimen,', readme_font),
    ('measurement, and instrument are all built here.', readme_font),
    ('Not an official CDISC product.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f'ReadMe: {len(readme_lines)} lines')

ReadMe: 83 lines


In [22]:
# ── Test_Identity sheet ──
ws_ti = wb.create_sheet('Test_Identity')

TI_FILLS = {
    'TESTCD':                          GREY_HEADER,
    'NCIt_Code':                       GREY_HEADER,
    'In_Scope_Domains':                GREY_HEADER,
    'SDTM_Domains':                    GREY_HEADER,
    'ct_source':                       GREY_HEADER,
    'TEST':                            GREEN_HEADER,
    'NCIt_Preferred_Term':             GREEN_HEADER,
    'NCIt_Synonyms':                   GREEN_HEADER,
    'NCIt_Definition':                 GREEN_HEADER,
    'UMLS_CUI':                        GREEN_HEADER,
    'NCIm_CUI':                        GREEN_HEADER,
    'Codelist_TESTCD':                 GREY_HEADER,
    'Instrument_NCIt_Code':            CHOCOLATE_HEADER,
    'Instrument_NCIt_Preferred_Term':  CHOCOLATE_HEADER,
    'Instrument_Match_Method':         GREY_HEADER,
    'Container_NCIt_Code':             COPPER_HEADER,
    'Container_NCIt_Preferred_Term':   COPPER_HEADER,
    'Has_DSS':                         YELLOW_HEADER,
    'DSS_Count':                       YELLOW_HEADER,
    'BC_Count':                        YELLOW_HEADER,
    'DS_Codes':                        YELLOW_HEADER,
    'BC_IDs':                          YELLOW_HEADER,
    'NCIt_Code_Conflict':              GREY_HEADER,
    'NCIt_Reference_Disagree':         GREY_HEADER,
}

TI_WIDTHS = {
    'TESTCD': 14, 'NCIt_Code': 12, 'In_Scope_Domains': 14, 'SDTM_Domains': 14,
    'ct_source': 18,
    'TEST': 38, 'NCIt_Preferred_Term': 38, 'NCIt_Synonyms': 50,
    'NCIt_Definition': 60, 'UMLS_CUI': 12, 'NCIm_CUI': 12,
    'Codelist_TESTCD': 14,
    'Instrument_NCIt_Code': 14, 'Instrument_NCIt_Preferred_Term': 38,
    'Instrument_Match_Method': 16,
    'Container_NCIt_Code': 14, 'Container_NCIt_Preferred_Term': 38,
    'Has_DSS': 9, 'DSS_Count': 9, 'BC_Count': 9, 'DS_Codes': 30, 'BC_IDs': 30,
    'NCIt_Code_Conflict': 12, 'NCIt_Reference_Disagree': 14,
}

write_sheet(ws_ti, ti_final, TI_FILLS, TI_WIDTHS)
print(f'Test_Identity: {len(ti_final):,} rows x {len(ti_final.columns)} cols')

Test_Identity: 10,309 rows x 24 cols


In [23]:
# ── Measurement_Specs sheet ──
ws_ms = wb.create_sheet('Measurement_Specs')

# Default everything to yellow; override greys for keys/flags, green for test identity,
# chocolate for instrument anchor, copper for container anchor.
MS_FILLS = {c: YELLOW_HEADER for c in ms_final.columns}
MS_FILLS.update({
    'domain':                          GREY_HEADER,
    'Observation_Class':               GREY_HEADER,
    'ds_id':                           GREY_HEADER,
    'bc_id':                           GREY_HEADER,
    'parent_bc_id':                    GREY_HEADER,
    'ct_source':                       GREY_HEADER,
    'TESTCD':                          GREEN_HEADER,
    'TEST':                            GREEN_HEADER,
    'TESTCD_NCIt':                     GREEN_HEADER,
    'Codelist_TESTCD':                 GREY_HEADER,
    'Instrument_NCIt_Code':            CHOCOLATE_HEADER,
    'Instrument_NCIt_Preferred_Term':  CHOCOLATE_HEADER,
    'Instrument_Match_Method':         GREY_HEADER,
    'Container_NCIt_Code':             COPPER_HEADER,
    'Container_NCIt_Preferred_Term':   COPPER_HEADER,
})

MS_WIDTHS = {
    'domain': 8, 'Observation_Class': 14, 'ds_id': 16, 'bc_id': 14,
    'parent_bc_id': 14, 'ct_source': 18,
    'TESTCD': 14, 'TEST': 38, 'TESTCD_NCIt': 12, 'Codelist_TESTCD': 14,
    'bc_short_name': 32, 'bc_definition': 50, 'bc_categories': 35,
    'bc_parent_label': 32, 'bc_hierarchy_path': 50, 'bc_type': 12,
    'result_scales': 18, 'bc_ncit_code': 12,
    'ds_short_name': 38, 'sdtmig_start_version': 12, 'sdtmig_end_version': 12,
    'package_date': 12, 'source': 16,
    'Instrument_NCIt_Code': 14, 'Instrument_NCIt_Preferred_Term': 38,
    'Instrument_Match_Method': 16,
    'Container_NCIt_Code': 14, 'Container_NCIt_Preferred_Term': 38,
    'LOINC_DSS': 14, 'LOINC_BC': 26,
    'CAT_value': 22, 'CAT_ncit': 12, 'CAT_codelist': 14,
    'SCAT_value': 22, 'SCAT_ncit': 12, 'SCAT_codelist': 14,
    'ORRESU_value': 12, 'ORRESU_ncit': 12, 'ORRESU_codelist': 14,
    'STRESU_value': 12, 'STRESU_ncit': 12, 'STRESU_codelist': 14,
    'EVAL_value': 22, 'EVAL_ncit': 12, 'EVAL_codelist': 14,
    'EVINTX_value': 22, 'EVINTX_ncit': 12, 'EVINTX_codelist': 14,
    'EVLINT_value': 22, 'EVLINT_ncit': 12, 'EVLINT_codelist': 14,
    'METHOD_value': 22, 'METHOD_ncit': 12, 'METHOD_codelist': 14,
    'LOC_value': 22, 'LOC_ncit': 12, 'LOC_codelist': 14,
    'ANTXLO_value': 12, 'ANTXLO_ncit': 12, 'ANTXLO_codelist': 14,
    'ANTXHI_value': 12, 'ANTXHI_ncit': 12, 'ANTXHI_codelist': 14,
    'ANVLLO_value': 12, 'ANVLLO_ncit': 12, 'ANVLLO_codelist': 14,
    'ANVLHI_value': 12, 'ANVLHI_ncit': 12, 'ANVLHI_codelist': 14,
    'Allowed_Units': 50,
}

write_sheet(ws_ms, ms_final, MS_FILLS, MS_WIDTHS)
print(f'Measurement_Specs: {len(ms_final):,} rows x {len(ms_final.columns)} cols')

Measurement_Specs: 175 rows x 70 cols


In [24]:
# ── BC_Categories sheet ──
ws_bcc = wb.create_sheet('BC_Categories')

BC_CATS_FILLS = {
    'bc_id':         GREY_HEADER,
    'bc_short_name': YELLOW_HEADER,
    'bc_type':       GREY_HEADER,
    'category':      YELLOW_HEADER,
    'domains':       YELLOW_HEADER,
    'in_dss_scope':  GREY_HEADER,
}

BC_CATS_WIDTHS = {
    'bc_id': 14, 'bc_short_name': 38, 'bc_type': 14, 'category': 35,
    'domains': 14, 'in_dss_scope': 12,
}

write_sheet(ws_bcc, bc_cats_final, BC_CATS_FILLS, BC_CATS_WIDTHS)
print(f'BC_Categories: {len(bc_cats_final):,} rows x {len(bc_cats_final.columns)} cols')

BC_Categories: 1,906 rows x 6 cols


In [25]:
# ── BC_Parents sheet ──
ws_bcp = wb.create_sheet('BC_Parents')

BC_PAR_FILLS = {
    'bc_id':                GREY_HEADER,
    'bc_short_name':        YELLOW_HEADER,
    'bc_type':              GREY_HEADER,
    'parent_bc_id':         GREY_HEADER,
    'parent_bc_short_name': YELLOW_HEADER,
    'parent_bc_type':       GREY_HEADER,
    'in_dss_scope':         GREY_HEADER,
}

BC_PAR_WIDTHS = {
    'bc_id': 14, 'bc_short_name': 38, 'bc_type': 14,
    'parent_bc_id': 14, 'parent_bc_short_name': 38, 'parent_bc_type': 14,
    'in_dss_scope': 12,
}

write_sheet(ws_bcp, bc_par_final, BC_PAR_FILLS, BC_PAR_WIDTHS)
print(f'BC_Parents: {len(bc_par_final):,} rows x {len(bc_par_final.columns)} cols')

BC_Parents: 329 rows x 7 cols


In [26]:
wb.save(OUTPUT_FILE)
print(f'\nWritten: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')


Written: /sessions/blissful-eloquent-meitner/mnt/cdisc-for-ai/sdtm-findings-graph/machine_actionable/Instrument_Findings.xlsx
File size: 1541 KB


## 8. Summary

In [27]:
print('=== Instrument_Findings summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'Test_Identity:    {len(ti_final):>5,} rows x {len(ti_final.columns):>2} cols')
print(f'  Has_DSS=Yes (COSMoS-pinned):       {(ti_final["Has_DSS"] == "Yes").sum():>5,}')
print(f'  Has_DSS=No  (coverage gap):        {(ti_final["Has_DSS"] == "No").sum():>5,}')
print(f'  ct_source distribution:')
for k, v in ti_final['ct_source'].value_counts().items():
    print(f'    {k:25s}  {v:>5,}')
print()
print(f'Measurement_Specs: {len(ms_final):>5,} rows x {len(ms_final.columns):>2} cols')
print(f'  By domain:')
for d in SCOPE_DOMAINS:
    n = (ms_final['domain'] == d).sum()
    print(f'    {d:>3s}  {n:>5,}')
print(f'  Instrument_Match_Method:')
for k, v in ms_final['Instrument_Match_Method'].value_counts(dropna=False).items():
    label = k if k else '(empty)'
    print(f'    {label:25s}  {v:>5,}')
print()
print(f'BC_Categories:    {len(bc_cats_final):>5,} rows x {len(bc_cats_final.columns):>2} cols')
print(f'  Distinct BCs:                  {bc_cats_final["bc_id"].nunique():>5,}')
print(f'  Distinct categories:           {bc_cats_final["category"].nunique():>5,}')
print()
print(f'BC_Parents:       {len(bc_par_final):>5,} rows x {len(bc_par_final.columns):>2} cols')
print(f'  Distinct child BCs:            {bc_par_final["bc_id"].nunique():>5,}')
print(f'  Distinct parent BCs:           {bc_par_final["parent_bc_id"].nunique():>5,}')

=== Instrument_Findings summary ===
Output: sdtm-findings-graph/machine_actionable/Instrument_Findings.xlsx

Test_Identity:    10,309 rows x 24 cols
  Has_DSS=Yes (COSMoS-pinned):         169
  Has_DSS=No  (coverage gap):        10,140
  ct_source distribution:
    instrument_codelist        10,257
    domain_codelist               48
    lzzt_example                   4

Measurement_Specs:   175 rows x 70 cols
  By domain:
     QS     17
     FT     23
     RS    135
  Instrument_Match_Method:
    exact                         97
    not_applicable                25
    normalized                    18
    suffix-strip                  18
    UNMATCHED                     17

BC_Categories:    1,906 rows x  6 cols
  Distinct BCs:                    325
  Distinct categories:             139

BC_Parents:         329 rows x  7 cols
  Distinct child BCs:              329
  Distinct parent BCs:              45
